In [2]:
!pip install scikit-image

   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
    --------------------------------------- 0.3/11.9 MB ? eta -:--:--
    --------------------------------------- 0.3/11.9 MB ? eta -:--:--
    ----------------


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

from skimage.feature import graycomatrix, graycoprops, local_binary_pattern

In [ ]:
DATASET_DIR = r"masukan_alamat_direktori"

In [5]:
def color_histogram_rgb(img, bins=32):
    features = []
    for i in range(3):  # R, G, B
        hist = cv2.calcHist([img], [i], None, [bins], [0,256])
        hist = cv2.normalize(hist, hist).flatten()
        features.extend(hist)
    return features

In [6]:
def color_histogram_hsv(img, bins=32):
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    features = []
    for i in range(3):  # H, S, V
        hist = cv2.calcHist([hsv], [i], None, [bins], [0,256])
        hist = cv2.normalize(hist, hist).flatten()
        features.extend(hist)
    return features

In [7]:
def glcm_features(img_gray):
    glcm = graycomatrix(
        img_gray,
        distances=[1],
        angles=[0],
        levels=256,
        symmetric=True,
        normed=True
    )

    features = [
        graycoprops(glcm, 'contrast')[0,0],
        graycoprops(glcm, 'dissimilarity')[0,0],
        graycoprops(glcm, 'homogeneity')[0,0],
        graycoprops(glcm, 'energy')[0,0],
        graycoprops(glcm, 'correlation')[0,0]
    ]
    return features

In [8]:
def lbp_features(img_gray, P=8, R=1):
    lbp = local_binary_pattern(img_gray, P, R, method="uniform")
    hist, _ = np.histogram(lbp, bins=np.arange(0, P+3), density=True)
    return hist.tolist()

In [9]:
def gabor_features(img_gray):
    features = []
    for theta in [0, np.pi/4, np.pi/2]:
        kernel = cv2.getGaborKernel(
            (21,21), 8.0, theta, 10.0, 0.5, 0, ktype=cv2.CV_32F
        )
        filtered = cv2.filter2D(img_gray, cv2.CV_8UC3, kernel)
        features.append(filtered.mean())
        features.append(filtered.var())
    return features

In [10]:
def shape_features(img_gray):
    _, thresh = cv2.threshold(img_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if len(contours) == 0:
        return [0,0,0,0]

    cnt = max(contours, key=cv2.contourArea)

    area = cv2.contourArea(cnt)
    perimeter = cv2.arcLength(cnt, True)

    compactness = (perimeter**2) / (4*np.pi*area) if area != 0 else 0

    x,y,w,h = cv2.boundingRect(cnt)
    aspect_ratio = w / h if h != 0 else 0

    return [area, perimeter, compactness, aspect_ratio]

In [11]:
def extract_features(img_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    features = []
    features.extend(color_histogram_rgb(img))
    features.extend(color_histogram_hsv(img))
    features.extend(glcm_features(img_gray))
    features.extend(lbp_features(img_gray))
    features.extend(gabor_features(img_gray))
    features.extend(shape_features(img_gray))

    return features

In [12]:
data = []
labels = []

for class_name in os.listdir(DATASET_DIR):
    class_path = os.path.join(DATASET_DIR, class_name)

    for img_name in tqdm(os.listdir(class_path), desc=class_name):
        img_path = os.path.join(class_path, img_name)
        feat = extract_features(img_path)

        data.append(feat)
        labels.append(class_name)

df = pd.DataFrame(data)
df["label"] = labels

df.to_csv("fitur_daun_tomat_train.csv", index=False)

yellow_leaf_curl_virus: 100%|████████████████████████████████████████████████████████| 640/640 [00:17<00:00, 35.96it/s]


In [13]:
df.head()
print("Jumlah sampel:", df.shape[0])
print("Jumlah fitur:", df.shape[1]-1)

Jumlah sampel: 5681
Jumlah fitur: 217


In [ ]:
rgb_count = len(color_histogram_rgb(np.zeros((224,224,3), dtype=np.uint8)))
hsv_count = len(color_histogram_hsv(np.zeros((224,224,3), dtype=np.uint8)))
glcm_count = len(glcm_features(np.zeros((224,224), dtype=np.uint8)))
lbp_count = len(lbp_features(np.zeros((224,224), dtype=np.uint8)))
gabor_count = len(gabor_features(np.zeros((224,224), dtype=np.uint8)))
shape_count = len(shape_features(np.zeros((224,224), dtype=np.uint8)))

warna_total = rgb_count + hsv_count
tekstur_total = glcm_count + lbp_count + gabor_count
bentuk_total = shape_count

total_fitur = warna_total + tekstur_total + bentuk_total

summary = pd.DataFrame([
    ["Fitur Warna", "Color Histogram (RGB)", rgb_count,
     "Distribusi intensitas RGB"],

    ["", "Color Histogram (HSV)", hsv_count,
     "Distribusi intensitas HSV"],

    ["Subtotal Warna", "", warna_total, ""],

    ["Fitur Tekstur", "GLCM", glcm_count,
     "Contrast, dissimilarity, homogeneity, energy, correlation"],

    ["", "LBP", lbp_count,
     "Local Binary Pattern Histogram"],

    ["", "Gabor", gabor_count,
     "Mean dan variance respon filter Gabor"],

    ["Subtotal Tekstur", "", tekstur_total, ""],

    ["Fitur Bentuk", "Shape Features", shape_count,
     "Area, perimeter, compactness, aspect ratio"],

    ["Subtotal Bentuk", "", bentuk_total, ""],

    ["TOTAL FITUR", "", total_fitur, ""]
],
columns=[
    "Kategori Fitur",
    "Metode/Teknik",
    "Jumlah Fitur",
    "Deskripsi"
])

print(summary.to_string(index=False))